In [ ]:
import json

# --- Configuration ---
DATABASE_FILE = "./data/train_data.txt"
OUTPUT_JSONL_FILE = "./data/train_data.jsonl" # Renamed for clarity

def parse_database_file_to_nested_jsonl(input_filepath, output_filepath):
    """
    Parses a 3-part text file and writes it to a structured, nested JSONL file.
    
    Input Format (in database.txt):
    - Lines starting with '#' are ignored.
    - Data lines: Messy Input | Clean Part 1 | Clean Part 2

    Output Format (in the .jsonl file):
    - Each line is a JSON object: 
      {"input": "Messy Input", "output": {"line1": "Clean Part 1", "line2": "Clean Part 2"}}
    """
    record_count = 0
    print(f"Reading data from '{input_filepath}' and writing to '{output_filepath}'...")

    try:
        # We open both files at once to stream data directly, which is memory-efficient
        with open(input_filepath, "r", encoding="utf-8") as infile, \
             open(output_filepath, 'w', encoding='utf-8') as outfile:
            
            for line in infile:
                # 1. Clean and check the line
                clean_line = line.strip()
                if not clean_line or clean_line.startswith("#"):
                    # This handles your first condition: ignore comments and empty lines
                    continue

                # 2. Split the line into three parts
                parts = clean_line.split("|")
                if len(parts) == 3:
                    # Strip whitespace from each part to be safe
                    input_address = parts[0].strip()
                    output_part1 = parts[1].strip()
                    output_part2 = parts[2].strip()

                    # 3. Create the dictionary in the new nested format
                    record = {
                        "input": input_address,
                        "output": {
                            "line1": output_part1,
                            "line2": output_part2
                        }
                    }

                    # 4. Write the JSON object as a new line in the output file
                    #    ensure_ascii=False writes Chinese characters directly, making the file readable.
                    outfile.write(json.dumps(record, ensure_ascii=False) + "\n")
                    record_count += 1
                else:
                    print(f"⚠️ WARNING: Skipping malformed line (did not have 3 parts): {clean_line}")

        print(f"✅ Successfully processed {record_count} records.")
        print(f"✅ Output saved to '{output_filepath}'.")

        # --- Verification Step to show the result ---
        print("\n--- Verifying Output File ---")
        with open(output_filepath, 'r', encoding='utf-8') as f:
            print("First 2 lines of the output file:")
            for i, line in enumerate(f):
                if i >= 2:
                    break
                # Use repr() to show the exact string content, including quotes
                print(f"Line {i}: {repr(line)}")

    except FileNotFoundError:
        print(f"❌ ERROR: The input file '{input_filepath}' was not found.")
    except Exception as e:
        print(f"❌ An unexpected error occurred: {e}")

# --- Main Execution ---
if __name__ == "__main__":
    parse_database_file_to_nested_jsonl(DATABASE_FILE, OUTPUT_JSONL_FILE)